# Prepare a reproducible appointment dataset

Use synthetic appointment records to validate an ingestion contract, avoid group leakage, check feature readiness and describe the resulting snapshot with a manifest.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Validate appointment records

Validate appointment record IDs and outcomes.

Implement appointment_errors(row). Report invalid_id for a missing, nonstring or blank ID. Then report missing_outcome when absent or invalid_outcome unless its value is integer 0 or 1, excluding booleans. Preserve this error order.


In [ ]:
def appointment_errors(row):
    raise NotImplementedError


In [ ]:
assert appointment_errors({"appointment_id":"A","outcome":0,"extra":1}) == []
assert appointment_errors({"appointment_id":" ","outcome":True}) == ["invalid_id","invalid_outcome"]
assert appointment_errors({}) == ["invalid_id","missing_outcome"]
assert appointment_errors({"appointment_id":1,"outcome":2}) == ["invalid_id","invalid_outcome"]
assert appointment_errors({"appointment_id":"A","outcome":None}) == ["invalid_outcome"], "A present but null outcome is invalid_outcome, not missing_outcome"
assert appointment_errors({"appointment_id":"A","outcome":1.0}) == ["invalid_outcome"], "The outcome must be the integer 0 or 1: reject the float 1.0"
print("OK")


## 2. Apply a trusted group assignment

Assign related records to one split.

Implement partition_groups(rows, assignment). Return train, validation and test lists, including empty ones. Assign each row by its group, preserve input order within each split and reject missing assignments or unknown split names.


In [ ]:
def partition_groups(rows, assignment):
    raise NotImplementedError


In [ ]:
rows = [{"group":"x","id":1},{"group":"y","id":2},{"group":"x","id":3}]
assert partition_groups(rows,{"x":"train","y":"test"}) == {"train":[rows[0],rows[2]],"validation":[],"test":[rows[1]]}
assert partition_groups([], {}) == {"train":[],"validation":[],"test":[]}
for assignments in ({}, {"x":"future"}):
    try:
        partition_groups(rows[:1], assignments)
    except ValueError:
        pass
    else:
        raise AssertionError("Reject invalid assignments")
print("OK")


## 3. Select an eligible historical value

Select history available at prediction time.

Implement available_value(rows, entity, decision). Rows are (entity, event_time, available_time, revision_id, value), with distinct comparable IDs for ties. Filter on entity and both timestamps. Choose maximum (event_time, available_time, revision_id), else None.


In [ ]:
def available_value(rows, entity, decision):
    raise NotImplementedError


In [ ]:
rows = [("a",1,2,"x",4),("a",3,8,"y",9),("b",4,4,"z",100),("a",2,5,"w",0)]
assert available_value(rows,"a",4) == 4
assert available_value(rows,"a",5) == 0
assert available_value(rows,"a",8) == 9
assert available_value(rows,"a",0) is None
assert available_value([("a",1,1,"b",2),("a",1,1,"a",3)],"a",1) == 2
assert available_value([("a",1,5,"r",1),("a",2,3,"s",2)],"a",5) == 2, "Choose the latest event_time first; available_time only breaks ties"
print("OK")


## 4. Fingerprint a JSON manifest

Hash a complete dataset manifest for later comparison.

Implement manifest_digest(manifest) for JSON data with string keys. Serialize with sorted keys, compact separators, unescaped Unicode, and disallow nonfinite numeric values. Return lowercase SHA-256 of UTF-8 bytes. Preserve list order.


In [ ]:
def manifest_digest(manifest):
    raise NotImplementedError


In [ ]:
import hashlib
assert manifest_digest({"b":2,"a":1}) == hashlib.sha256(b'{"a":1,"b":2}').hexdigest()
assert manifest_digest({"schema":1,"split":"train"}) != manifest_digest({"schema":1,"split":"test"})
assert manifest_digest({"ids":["a","b"]}) != manifest_digest({"ids":["b","a"]})
assert manifest_digest({"city":"Québec"}) == hashlib.sha256('{"city":"Québec"}'.encode()).hexdigest()
try:
    manifest_digest({"x":float("nan")})
except ValueError:
    pass
else:
    raise AssertionError("reject NaN")
print("OK")


## 5. Validate, split and fingerprint a snapshot

Implement prepare_snapshot(rows, assignment, schema_version). Each row has appointment_id,
outcome and group fields. Collect contract failure reason counts and split only accepted rows.
This fixture assumes accepted IDs are unique and group is valid. Build a manifest with schema_version
and a splits mapping from split name to appointment IDs in split order, then return
(splits, rejection reason counts, manifest digest). Reuse the three existing helpers.


In [ ]:
def prepare_snapshot(rows, assignment, schema_version):
    raise NotImplementedError


In [ ]:
rows = [{"appointment_id":"a","outcome":0,"group":"x"},{"appointment_id":"b","outcome":True,"group":"unknown"},{"appointment_id":"c","outcome":1,"group":"y"}]
splits, reasons, digest = prepare_snapshot(rows, {"x":"train","y":"test"}, 2)
assert reasons == {"invalid_outcome":1} and splits["train"] == [rows[0]] and splits["test"] == [rows[2]]
assert digest == manifest_digest({"schema_version":2,"splits":{"train":["a"],"validation":[],"test":["c"]}})
assert prepare_snapshot(rows, {"x":"test","y":"train"}, 2)[2] != digest
assert prepare_snapshot(rows, {"x":"train","y":"test"}, 3)[2] != digest
print("OK")


## Interpretation

Write two deployment scenarios that call for different split rules. Explain which rows or features your pipeline would quarantine and what version would let a colleague reproduce the split.

**Your answer**
